In [0]:
spark


In [0]:
%sql
USE siddarthasamisetty.enterprise_de_project;


Read Product Master from Volume

In [0]:
product_raw_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("/Volumes/siddarthasamisetty/enterprise_de_project/source_data/Usecase2/product_master.csv")
)

product_raw_df.show(5)


+----------+------------+---------+----------+
|product_id|product_name| category|unit_price|
+----------+------------+---------+----------+
|       101| Product_101| Clothing|     38913|
|       102| Product_102|Furniture|     12154|
|       103| Product_103| Clothing|     15230|
|       104| Product_104| Clothing|      9567|
|       105| Product_105|Furniture|     46815|
+----------+------------+---------+----------+
only showing top 5 rows


Add Bronze Metadata Columns

In [0]:
from pyspark.sql.functions import current_timestamp, lit

product_bronze_df = (
    product_raw_df
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("source_system", lit("local_csv"))
)


Write Bronze Product Table (Delta)

In [0]:
(
    product_bronze_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("bronze_product_master")
)


Verify Bronze Table

In [0]:
%sql
SELECT * FROM bronze_product_master LIMIT 5;


product_id,product_name,category,unit_price,ingestion_timestamp,source_system
101,Product_101,Clothing,38913,2025-12-22T10:50:53.088Z,local_csv
102,Product_102,Furniture,12154,2025-12-22T10:50:53.088Z,local_csv
103,Product_103,Clothing,15230,2025-12-22T10:50:53.088Z,local_csv
104,Product_104,Clothing,9567,2025-12-22T10:50:53.088Z,local_csv
105,Product_105,Furniture,46815,2025-12-22T10:50:53.088Z,local_csv


Bronze Store / Region Table

In [0]:
store_raw_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("/Volumes/siddarthasamisetty/enterprise_de_project/source_data/Usecase2/store_region.csv")
)

store_bronze_df = (
    store_raw_df
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("source_system", lit("local_csv"))
)

(
    store_bronze_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("bronze_store_region")
)


In [0]:
%sql
SELECT * FROM bronze_store_region LIMIT 5;


store_id,store_name,region,ingestion_timestamp,source_system
1,Bangalore Store,South,2025-12-22T10:53:37.466Z,local_csv
2,Chennai Store,South,2025-12-22T10:53:37.466Z,local_csv
3,Hyderabad Store,South,2025-12-22T10:53:37.466Z,local_csv
4,Coimbatore Store,South,2025-12-22T10:53:37.466Z,local_csv
5,Mysore Store,South,2025-12-22T10:53:37.466Z,local_csv


Bronze Sales Transactions Table

In [0]:
sales_raw_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("/Volumes/siddarthasamisetty/enterprise_de_project/source_data/Usecase2/sales_transactions.csv")
)

sales_bronze_df = (
    sales_raw_df
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("source_system", lit("local_csv"))
)

(
    sales_bronze_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("bronze_sales_transactions")
)


In [0]:
%sql
SELECT COUNT(*) FROM bronze_sales_transactions;


COUNT(*)
1000


In [0]:
# ============================================================
# STEP 4.2 — SILVER PRODUCT MASTER (Cleansing & Schema Enforced)
# ============================================================

from pyspark.sql.functions import col, when, lit

# 1. Read Bronze Product Master
bronze_product_df = spark.table("bronze_product_master")

# 2. Remove duplicate records (based on product_id)
silver_product_dedup_df = bronze_product_df.dropDuplicates(["product_id"])

# 3. Remove invalid records (nulls and invalid prices)
silver_product_valid_df = (
    silver_product_dedup_df
    .filter(col("product_id").isNotNull())
    .filter(col("product_name").isNotNull())
    .filter(col("unit_price").isNotNull())
    .filter(col("unit_price") > 0)
)

# 4. Handle optional null values
silver_product_clean_df = (
    silver_product_valid_df
    .withColumn(
        "category",
        when(col("category").isNull(), lit("UNKNOWN"))
        .otherwise(col("category"))
    )
)

# 5. Select business-ready schema (remove technical columns)
silver_product_final_df = silver_product_clean_df.select(
    "product_id",
    "product_name",
    "category",
    "unit_price"
)

# 6. Write Silver Product Master as Delta table (idempotent)
(
    silver_product_final_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("silver_product_master")
)

# 7. Quick validation
display(silver_product_final_df.limit(10))


product_id,product_name,category,unit_price
101,Product_101,Clothing,38913
102,Product_102,Furniture,12154
103,Product_103,Clothing,15230
104,Product_104,Clothing,9567
105,Product_105,Furniture,46815
106,Product_106,Furniture,21275
107,Product_107,Furniture,59882
108,Product_108,Clothing,1251
109,Product_109,Electronics,52076
110,Product_110,Furniture,51236


SILVER STORE / REGION

In [0]:
# ============================================================
# STEP 4.3 — SILVER STORE / REGION (Cleansing & Schema Enforced)
# ============================================================

from pyspark.sql.functions import col, when, lit

# 1. Read Bronze Store Region
bronze_store_df = spark.table("bronze_store_region")

# 2. Remove duplicate records (based on store_id)
silver_store_dedup_df = bronze_store_df.dropDuplicates(["store_id"])

# 3. Remove invalid records (null store_id or store_name)
silver_store_valid_df = (
    silver_store_dedup_df
    .filter(col("store_id").isNotNull())
    .filter(col("store_name").isNotNull())
)

# 4. Handle optional null values
silver_store_clean_df = (
    silver_store_valid_df
    .withColumn(
        "region",
        when(col("region").isNull(), lit("UNKNOWN"))
        .otherwise(col("region"))
    )
)

# 5. Select business-ready schema
silver_store_final_df = silver_store_clean_df.select(
    "store_id",
    "store_name",
    "region"
)

# 6. Write Silver Store Region table (Delta, idempotent)
(
    silver_store_final_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("silver_store_region")
)

# 7. Quick validation
display(silver_store_final_df.limit(10))


store_id,store_name,region
1,Bangalore Store,South
2,Chennai Store,South
3,Hyderabad Store,South
4,Coimbatore Store,South
5,Mysore Store,South
6,Delhi Store,North
7,Noida Store,North
8,Gurgaon Store,North
9,Chandigarh Store,North
10,Jaipur Store,North


SILVER SALES TRANSACTIONS

In [0]:
# ============================================================
# STEP 4.4 — SILVER SALES (Cleansing, Calibration & Quarantine)
# ============================================================

from pyspark.sql.functions import col, lit, when, round

# -------------------------------
# 1. Read Tables with Aliases
# -------------------------------
sales = spark.table("bronze_sales_transactions").alias("sales")
product = spark.table("silver_product_master").alias("product")
store = spark.table("silver_store_region").alias("store")

# -------------------------------
# 2. Join with Reference Data
# -------------------------------
sales_enriched_df = (
    sales
    .join(product, col("sales.product_id") == col("product.product_id"), "left")
    .join(store, col("sales.store_id") == col("store.store_id"), "left")
)

# -------------------------------
# 3. Apply Data Quality Rules
# -------------------------------
sales_validated_df = (
    sales_enriched_df
    .withColumn(
        "rejection_reason",
        when(col("sales.quantity") <= 0, "INVALID_QUANTITY")
        .when(col("sales.unit_price") <= 0, "INVALID_UNIT_PRICE")
        .when(col("product.product_id").isNull(), "INVALID_PRODUCT_ID")
        .when(col("store.store_id").isNull(), "INVALID_STORE_ID")
        .otherwise(None)
    )
)

# -------------------------------
# 4. Separate Valid & Invalid
# -------------------------------
valid_sales_df = sales_validated_df.filter(col("rejection_reason").isNull())
invalid_sales_df = sales_validated_df.filter(col("rejection_reason").isNotNull())

# -------------------------------
# 5. Data Calibration (MANDATORY)
# total_amount = quantity * unit_price - discount
# -------------------------------
silver_sales_calibrated_df = (
    valid_sales_df
    .withColumn(
        "total_amount",
        round(
            col("sales.quantity") * col("sales.unit_price") - col("sales.discount"),
            2
        )
    )
)

# -------------------------------
# 6. Final Silver Sales Schema
# -------------------------------
silver_sales_final_df = silver_sales_calibrated_df.select(
    col("sales.transaction_id").alias("transaction_id"),
    col("sales.transaction_date").alias("transaction_date"),
    col("sales.product_id").alias("product_id"),
    col("sales.store_id").alias("store_id"),
    col("sales.quantity").alias("quantity"),
    col("sales.unit_price").alias("unit_price"),
    col("sales.discount").alias("discount"),
    col("total_amount")
)

# -------------------------------
# 7. Write SILVER SALES TABLE
# -------------------------------
(
    silver_sales_final_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("silver_sales_transactions")
)

# -------------------------------
# 8. Write QUARANTINE TABLE
# -------------------------------
(
    invalid_sales_df.select(
        col("sales.transaction_id").alias("transaction_id"),
        col("sales.product_id").alias("product_id"),
        col("sales.store_id").alias("store_id"),
        col("sales.quantity").alias("quantity"),
        col("sales.unit_price").alias("unit_price"),
        col("sales.discount").alias("discount"),
        col("sales.total_amount").alias("original_total_amount"),
        col("rejection_reason")
    )
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("silver_sales_quarantine")
)

# -------------------------------
# 9. Validation
# -------------------------------
display(silver_sales_final_df.limit(10))
display(invalid_sales_df.limit(10))


transaction_id,transaction_date,product_id,store_id,quantity,unit_price,discount,total_amount
1,2024-01-02,131,5,5,38303,0,191515
2,2024-01-03,124,4,1,8983,500,8483
3,2024-01-04,120,4,1,40625,500,40125
5,2024-01-06,116,2,5,4568,1000,21840
6,2024-01-07,149,5,2,31232,0,62464
7,2024-01-08,149,4,2,20867,0,41734
8,2024-01-09,122,4,2,45726,1000,90452
9,2024-01-10,125,1,1,8926,1000,7926
10,2024-01-11,117,4,4,10707,0,42828
11,2024-01-12,114,4,5,1256,1000,5280


transaction_id,transaction_date,product_id,store_id,quantity,unit_price,discount,total_amount,ingestion_timestamp,source_system,product_id,product_name,category,unit_price,store_id,store_name,region,rejection_reason
4,2024-01-05,132,99,1,48616,1000,47616,2025-12-22T10:54:17.664Z,local_csv,132,Product_132,Clothing,28960,null,null,null,INVALID_STORE_ID
16,2024-01-17,149,99,1,19597,500,19097,2025-12-22T10:54:17.664Z,local_csv,149,Product_149,Furniture,48527,null,null,null,INVALID_STORE_ID
18,2024-01-19,133,99,4,52433,500,209232,2025-12-22T10:54:17.664Z,local_csv,133,Product_133,Furniture,45001,null,null,null,INVALID_STORE_ID
22,2024-01-23,109,99,3,2385,0,7155,2025-12-22T10:54:17.664Z,local_csv,109,Product_109,Electronics,52076,null,null,null,INVALID_STORE_ID
30,2024-01-01,144,99,5,49647,500,3552,2025-12-22T10:54:17.664Z,local_csv,144,Product_144,Clothing,28441,null,null,null,INVALID_STORE_ID
31,2024-01-02,117,99,2,856,0,1712,2025-12-22T10:54:17.664Z,local_csv,117,Product_117,Clothing,56777,null,null,null,INVALID_STORE_ID
40,2024-01-11,142,99,5,34244,0,226,2025-12-22T10:54:17.664Z,local_csv,142,Product_142,Furniture,20202,null,null,null,INVALID_STORE_ID
45,2024-01-16,132,99,4,27383,1000,108532,2025-12-22T10:54:17.664Z,local_csv,132,Product_132,Clothing,28960,null,null,null,INVALID_STORE_ID
55,2024-01-26,130,99,1,33560,1000,32560,2025-12-22T10:54:17.664Z,local_csv,130,Product_130,Clothing,34019,null,null,null,INVALID_STORE_ID
56,2024-01-27,103,99,4,28116,0,112464,2025-12-22T10:54:17.664Z,local_csv,103,Product_103,Clothing,15230,null,null,null,INVALID_STORE_ID


DELTA MERGE (Incremental UPSERT)

In [0]:
# ============================================================
# PHASE 5 — STEP 5.1: DELTA MERGE (Incremental UPSERT)
# ============================================================

from pyspark.sql import Row
from pyspark.sql.functions import col

# 1. Simulate NEW + UPDATED sales records (incremental batch)
incremental_sales_data = [
    # Existing transaction_id (UPDATE case)
    Row(transaction_id=1, transaction_date="2024-01-01", product_id=101, store_id=1,
        quantity=3, unit_price=60000, discount=1000, total_amount=179000),

    # New transaction_id (INSERT case)
    Row(transaction_id=9999, transaction_date="2024-01-05", product_id=102, store_id=2,
        quantity=1, unit_price=30000, discount=0, total_amount=30000)
]

incremental_sales_df = spark.createDataFrame(incremental_sales_data)

# 2. Create TEMP view for MERGE
incremental_sales_df.createOrReplaceTempView("incremental_sales")

# 3. MERGE INTO silver_sales_transactions
spark.sql("""
MERGE INTO silver_sales_transactions AS target
USING incremental_sales AS source
ON target.transaction_id = source.transaction_id

WHEN MATCHED THEN
  UPDATE SET
    target.transaction_date = source.transaction_date,
    target.product_id       = source.product_id,
    target.store_id         = source.store_id,
    target.quantity         = source.quantity,
    target.unit_price       = source.unit_price,
    target.discount         = source.discount,
    target.total_amount     = source.total_amount

WHEN NOT MATCHED THEN
  INSERT *
""")

# 4. Validation
display(
    spark.table("silver_sales_transactions")
    .filter(col("transaction_id").isin(1, 9999))
)


transaction_id,transaction_date,product_id,store_id,quantity,unit_price,discount,total_amount
1,2024-01-01,101,1,3,60000,1000,179000
9999,2024-01-05,102,2,1,30000,0,30000


DELTA SCHEMA EVOLUTION

In [0]:
# ============================================================
# PHASE 5 — STEP 5.2: DELTA SCHEMA EVOLUTION
# ============================================================

from pyspark.sql.functions import lit

# 1. Read existing Silver Sales table
silver_sales_df = spark.table("silver_sales_transactions")

# 2. Add NEW column (schema evolution)
silver_sales_evolved_df = (
    silver_sales_df
    .withColumn("data_source", lit("batch_ingestion"))
)

# 3. Write back with schema evolution enabled
(
    silver_sales_evolved_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver_sales_transactions")
)

# 4. Validation
display(silver_sales_evolved_df.limit(10))


transaction_id,transaction_date,product_id,store_id,quantity,unit_price,discount,total_amount,data_source
2,2024-01-03,124,4,1,8983,500,8483,batch_ingestion
3,2024-01-04,120,4,1,40625,500,40125,batch_ingestion
5,2024-01-06,116,2,5,4568,1000,21840,batch_ingestion
6,2024-01-07,149,5,2,31232,0,62464,batch_ingestion
7,2024-01-08,149,4,2,20867,0,41734,batch_ingestion
8,2024-01-09,122,4,2,45726,1000,90452,batch_ingestion
9,2024-01-10,125,1,1,8926,1000,7926,batch_ingestion
10,2024-01-11,117,4,4,10707,0,42828,batch_ingestion
11,2024-01-12,114,4,5,1256,1000,5280,batch_ingestion
12,2024-01-13,104,5,3,44465,500,132895,batch_ingestion


DELTA TIME TRAVEL

In [0]:
%sql
-- ============================================================
-- PHASE 5 — STEP 5.3: DELTA TIME TRAVEL
-- ============================================================

-- 1. View Delta table history (versions)
DESCRIBE HISTORY silver_sales_transactions;

-- 2. Query an older version (before schema evolution / merge)
-- Use VERSION number from history output (example: VERSION AS OF 0)
SELECT *
FROM silver_sales_transactions VERSION AS OF 0
LIMIT 10;

-- 3. Query current version for comparison
SELECT *
FROM silver_sales_transactions
LIMIT 10;


transaction_id,transaction_date,product_id,store_id,quantity,unit_price,discount,total_amount,data_source
2,2024-01-03,124,4,1,8983,500,8483,batch_ingestion
3,2024-01-04,120,4,1,40625,500,40125,batch_ingestion
5,2024-01-06,116,2,5,4568,1000,21840,batch_ingestion
6,2024-01-07,149,5,2,31232,0,62464,batch_ingestion
7,2024-01-08,149,4,2,20867,0,41734,batch_ingestion
8,2024-01-09,122,4,2,45726,1000,90452,batch_ingestion
9,2024-01-10,125,1,1,8926,1000,7926,batch_ingestion
10,2024-01-11,117,4,4,10707,0,42828,batch_ingestion
11,2024-01-12,114,4,5,1256,1000,5280,batch_ingestion
12,2024-01-13,104,5,3,44465,500,132895,batch_ingestion


DELTA VERSION ROLLBACK (RESTORE TABLE)

In [0]:
%sql
-- ============================================================
-- PHASE 5 — STEP 5.4: DELTA VERSION ROLLBACK
-- ============================================================

-- 1. Check available versions
DESCRIBE HISTORY silver_sales_transactions;

-- 2. Restore table to a previous version
-- (Use a valid version number from the history output, example: VERSION AS OF 1)
RESTORE TABLE silver_sales_transactions TO VERSION AS OF 1;

-- 3. Validate rollback
SELECT *
FROM silver_sales_transactions
LIMIT 10;


transaction_id,transaction_date,product_id,store_id,quantity,unit_price,discount,total_amount
1,2024-01-02,131,5,5,38303,0,191515
2,2024-01-03,124,4,1,8983,500,8483
3,2024-01-04,120,4,1,40625,500,40125
5,2024-01-06,116,2,5,4568,1000,21840
6,2024-01-07,149,5,2,31232,0,62464
7,2024-01-08,149,4,2,20867,0,41734
8,2024-01-09,122,4,2,45726,1000,90452
9,2024-01-10,125,1,1,8926,1000,7926
10,2024-01-11,117,4,4,10707,0,42828
11,2024-01-12,114,4,5,1256,1000,5280


DELTA OPTIMIZE (Performance Optimization)

In [0]:
%sql
-- ============================================================
-- PHASE 5 — STEP 5.5: DELTA OPTIMIZE
-- ============================================================

OPTIMIZE silver_sales_transactions;


path,metrics
abfss://unity-catalog-storage@dbstoragebvt5lrpd4v7la.dfs.core.windows.net/646459859347545/__unitystorage/catalogs/73e99aa5-0cd0-46c1-a350-fb696a23ce32/tables/6799c86d-ab5a-4899-8886-81a0aba4b27d,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, null, null, 0, 0, 1, 1, true, 0, 0, 1766402102623, 1766402103748, 8, 0, null, List(0, 0), null, 8, 8, 0, 0, null)"


PHASE 6 — STEP 6.1
GOLD: DAILY SALES SUMMARY

In [0]:
# ============================================================
# PHASE 6 — STEP 6.1: GOLD DAILY SALES SUMMARY
# ============================================================

from pyspark.sql.functions import col, sum as _sum, count as _count

# 1. Read trusted Silver sales data
silver_sales_df = spark.table("silver_sales_transactions")

# 2. Aggregate daily metrics
gold_daily_sales_df = (
    silver_sales_df
    .groupBy("transaction_date")
    .agg(
        _sum("total_amount").alias("total_revenue"),
        _count("transaction_id").alias("total_transactions")
    )
)

# 3. Write Gold table (Delta)
(
    gold_daily_sales_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("gold_daily_sales_summary")
)

# 4. Validation
display(gold_daily_sales_df.orderBy("transaction_date"))


transaction_date,total_revenue,total_transactions
2024-01-01,2258088,26
2024-01-02,2839314,28
2024-01-03,2515865,28
2024-01-04,2667068,31
2024-01-05,2331147,25
2024-01-06,1958626,23
2024-01-07,1871932,24
2024-01-08,2201487,31
2024-01-09,2949590,28
2024-01-10,2200614,28


PHASE 6 — STEP 6.2
GOLD: MONTHLY REVENUE BY REGION

In [0]:
# ============================================================
# PHASE 6 — STEP 6.2: GOLD MONTHLY REVENUE BY REGION
# ============================================================

from pyspark.sql.functions import col, sum as _sum, date_format

# 1. Read trusted Silver tables
silver_sales_df = spark.table("silver_sales_transactions")
silver_store_df = spark.table("silver_store_region")

# 2. Join sales with store to get region
sales_with_region_df = (
    silver_sales_df
    .join(silver_store_df, "store_id", "inner")
)

# 3. Derive Year-Month column
monthly_region_df = (
    sales_with_region_df
    .withColumn("year_month", date_format(col("transaction_date"), "yyyy-MM"))
    .groupBy("year_month", "region")
    .agg(
        _sum("total_amount").alias("monthly_revenue")
    )
)

# 4. Write Gold table
(
    monthly_region_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("gold_monthly_revenue_by_region")
)

# 5. Validation
display(monthly_region_df.orderBy("year_month", "region"))


year_month,region,monthly_revenue
2024-01,South,70051160


PHASE 6 — STEP 6.3
GOLD: PRODUCT PERFORMANCE METRICS

In [0]:
# ============================================================
# PHASE 6 — STEP 6.3: GOLD PRODUCT PERFORMANCE METRICS
# ============================================================

from pyspark.sql.functions import col, sum as _sum, count as _count

# 1. Read trusted Silver tables
silver_sales_df = spark.table("silver_sales_transactions")
silver_product_df = spark.table("silver_product_master")

# 2. Join sales with product dimension
sales_with_product_df = (
    silver_sales_df
    .join(silver_product_df, "product_id", "inner")
)

# 3. Aggregate product-level metrics
gold_product_performance_df = (
    sales_with_product_df
    .groupBy("product_id", "product_name", "category")
    .agg(
        _sum("total_amount").alias("total_revenue"),
        _sum("quantity").alias("total_quantity_sold"),
        _count("transaction_id").alias("total_transactions")
    )
)

# 4. Write Gold table
(
    gold_product_performance_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("gold_product_performance_metrics")
)

# 5. Validation
display(gold_product_performance_df.orderBy(col("total_revenue").desc()))


product_id,product_name,category,total_revenue,total_quantity_sold,total_transactions
147,Product_147,Clothing,2166183,66,23
112,Product_112,Furniture,2164615,62,21
131,Product_131,Clothing,2028505,53,18
132,Product_132,Clothing,2002286,59,19
110,Product_110,Furniture,1912390,68,22
111,Product_111,Clothing,1881543,61,19
129,Product_129,Electronics,1855149,65,18
108,Product_108,Clothing,1819948,49,16
102,Product_102,Furniture,1807519,54,18
145,Product_145,Clothing,1802505,59,22


PHASE 6 — STEP 6.4
GOLD PERFORMANCE OPTIMIZATION

In [0]:
%sql
-- ============================================================
-- PHASE 6 — STEP 6.4: OPTIMIZE GOLD TABLES
-- ============================================================

OPTIMIZE gold_daily_sales_summary;
OPTIMIZE gold_monthly_revenue_by_region;
OPTIMIZE gold_product_performance_metrics;


path,metrics
abfss://unity-catalog-storage@dbstoragebvt5lrpd4v7la.dfs.core.windows.net/646459859347545/__unitystorage/catalogs/73e99aa5-0cd0-46c1-a350-fb696a23ce32/tables/40b18fc7-9672-44e7-9642-7ed431568e7f,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, null, null, 0, 0, 1, 1, true, 0, 0, 1766402405525, 1766402405802, 8, 0, null, List(0, 0), null, 6, 6, 0, 0, null)"


In [0]:
# ============================================================
# DATA ENRICHMENT: ADD MULTI-REGION, MULTI-MONTH SALES
# ============================================================

from pyspark.sql import Row

new_sales_data = [
    # North Region (Feb)
    Row(transaction_id=20001, transaction_date="2024-02-05", product_id=101, store_id=4,
        quantity=2, unit_price=60000, discount=1000, total_amount=119000),
    
    # West Region (Feb)
    Row(transaction_id=20002, transaction_date="2024-02-10", product_id=102, store_id=3,
        quantity=1, unit_price=30000, discount=0, total_amount=30000),

    # East Region (Mar)
    Row(transaction_id=20003, transaction_date="2024-03-15", product_id=103, store_id=5,
        quantity=3, unit_price=4000, discount=500, total_amount=11500)
]

new_sales_df = spark.createDataFrame(new_sales_data)
new_sales_df.createOrReplaceTempView("new_sales")

spark.sql("""
MERGE INTO silver_sales_transactions AS target
USING new_sales AS source
ON target.transaction_id = source.transaction_id

WHEN NOT MATCHED THEN
  INSERT *
""")


DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
# ============================================================
# REBUILD GOLD MONTHLY REVENUE BY REGION
# ============================================================

from pyspark.sql.functions import col, sum as _sum, date_format

silver_sales_df = spark.table("silver_sales_transactions")
silver_store_df = spark.table("silver_store_region")

gold_monthly_region_df = (
    silver_sales_df
    .join(silver_store_df, "store_id", "inner")
    .withColumn("year_month", date_format(col("transaction_date"), "yyyy-MM"))
    .groupBy("year_month", "region")
    .agg(_sum("total_amount").alias("monthly_revenue"))
)

(
    gold_monthly_region_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("gold_monthly_revenue_by_region")
)

display(gold_monthly_region_df)


year_month,region,monthly_revenue
2024-01,South,70051160
2024-03,South,11500
2024-02,South,149000


In [0]:
# ============================================================
# FIX STORE DIMENSION — ADD MULTI-REGION STORES
# ============================================================

from pyspark.sql import Row

new_stores = [
    Row(store_id=10, store_name="Delhi Central", region="North"),
    Row(store_id=11, store_name="Mumbai Hub", region="West"),
    Row(store_id=12, store_name="Kolkata Plaza", region="East")
]

new_store_df = spark.createDataFrame(new_stores)
new_store_df.createOrReplaceTempView("new_stores")

spark.sql("""
MERGE INTO silver_store_region AS target
USING new_stores AS source
ON target.store_id = source.store_id

WHEN NOT MATCHED THEN
  INSERT *
""")


DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
# ============================================================
# ADD SALES FOR NEW REGIONS
# ============================================================

from pyspark.sql import Row

regional_sales = [
    Row(transaction_id=30001, transaction_date="2024-02-12", product_id=101, store_id=10,
        quantity=1, unit_price=60000, discount=0, total_amount=60000),

    Row(transaction_id=30002, transaction_date="2024-03-08", product_id=102, store_id=11,
        quantity=2, unit_price=30000, discount=0, total_amount=60000),

    Row(transaction_id=30003, transaction_date="2024-03-20", product_id=103, store_id=12,
        quantity=3, unit_price=4000, discount=0, total_amount=12000)
]

regional_sales_df = spark.createDataFrame(regional_sales)
regional_sales_df.createOrReplaceTempView("regional_sales")

spark.sql("""
MERGE INTO silver_sales_transactions AS target
USING regional_sales AS source
ON target.transaction_id = source.transaction_id

WHEN NOT MATCHED THEN
  INSERT *
""")


DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
# ============================================================
# REBUILD GOLD MONTHLY REVENUE BY REGION (FINAL)
# ============================================================

from pyspark.sql.functions import col, sum as _sum, date_format

silver_sales_df = spark.table("silver_sales_transactions")
silver_store_df = spark.table("silver_store_region")

gold_monthly_region_df = (
    silver_sales_df
    .join(silver_store_df, "store_id", "inner")
    .withColumn("year_month", date_format(col("transaction_date"), "yyyy-MM"))
    .groupBy("year_month", "region")
    .agg(_sum("total_amount").alias("monthly_revenue"))
)

(
    gold_monthly_region_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("gold_monthly_revenue_by_region")
)

display(gold_monthly_region_df.orderBy("year_month", "region"))


year_month,region,monthly_revenue
2024-01,South,70051160
2024-02,North,60000
2024-02,South,149000
2024-03,South,11500
2024-03,West,72000


PHASE 8 — STEP 8.1
Create Pipeline Logging Table

In [0]:

%sql
CREATE TABLE IF NOT EXISTS siddarthasamisetty.enterprise_de_project.pipeline_run_logs (
    pipeline_name STRING,
    layer STRING,
    step_name STRING,
    start_time TIMESTAMP,
    end_time TIMESTAMP,
    record_count BIGINT,
    status STRING,
    error_message STRING
)
USING DELTA;





PHASE 8 — STEP 8.2
Log Ingestion Start & End Times (Enterprise Pattern)

In [0]:
from datetime import datetime
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, LongType

pipeline_name = "enterprise_de_pipeline"
layer = "silver"
step_name = "silver_sales_processing"

start_time = datetime.now()

try:
    df = spark.table("siddarthasamisetty.enterprise_de_project.silver_sales_transactions")
    record_count = df.count()
    status = "SUCCESS"
    error_message = None
except Exception as e:
    record_count = None
    status = "FAILED"
    error_message = str(e)

end_time = datetime.now()

schema = StructType([
    StructField("pipeline_name", StringType(), True),
    StructField("layer", StringType(), True),
    StructField("step_name", StringType(), True),
    StructField("start_time", TimestampType(), True),
    StructField("end_time", TimestampType(), True),
    StructField("record_count", LongType(), True),
    StructField("status", StringType(), True),
    StructField("error_message", StringType(), True)
])

log_df = spark.createDataFrame([(
    pipeline_name, layer, step_name,
    start_time, end_time,
    record_count, status, error_message
)], schema)

log_df.write.format("delta").mode("append") \
    .saveAsTable("siddarthasamisetty.enterprise_de_project.pipeline_run_logs")

# Validate logs
display(
    spark.table("siddarthasamisetty.enterprise_de_project.pipeline_run_logs")
    .orderBy("start_time", ascending=False)
)



pipeline_name,layer,step_name,start_time,end_time,record_count,status,error_message
enterprise_de_pipeline,silver,silver_sales_processing,2025-12-22T13:42:13.583Z,2025-12-22T13:42:13.790Z,803,SUCCESS,null
enterprise_de_pipeline,gold,gold_record_count,2025-12-22T13:42:06.230Z,2025-12-22T13:42:06.825Z,30,SUCCESS,null
enterprise_de_pipeline,bronze,bronze_record_count,2025-12-22T13:42:06.230Z,2025-12-22T13:42:06.449Z,1000,SUCCESS,null
enterprise_de_pipeline,silver,silver_record_count,2025-12-22T13:42:06.230Z,2025-12-22T13:42:06.638Z,803,SUCCESS,null
enterprise_de_pipeline,silver,quarantine_record_count,2025-12-22T13:41:49.178Z,2025-12-22T13:41:49.178Z,203,SUCCESS,null
enterprise_de_pipeline,silver,rerun_safety_validation,2025-12-22T13:41:25.071Z,2025-12-22T13:41:29.127Z,803,SUCCESS,null
enterprise_de_pipeline,silver,quarantine_record_count,2025-12-22T13:39:56.589Z,2025-12-22T13:39:56.589Z,203,SUCCESS,null
enterprise_de_pipeline,silver,silver_record_count,2025-12-22T13:39:24.927Z,2025-12-22T13:39:25.430Z,803,SUCCESS,null
enterprise_de_pipeline,bronze,bronze_record_count,2025-12-22T13:39:24.927Z,2025-12-22T13:39:25.220Z,1000,SUCCESS,null
enterprise_de_pipeline,gold,gold_record_count,2025-12-22T13:39:24.927Z,2025-12-22T13:39:25.652Z,30,SUCCESS,null


PHASE 8 — STEP 8.3
LOG RECORD COUNTS PER LAYER (Bronze → Silver → Gold)

In [0]:
from datetime import datetime

layers = [
    ("bronze", "bronze_sales_transactions"),
    ("silver", "silver_sales_transactions"),
    ("gold", "gold_daily_sales_summary")
]

logs = []
start_time = datetime.now()

for layer, table in layers:
    count = spark.table(f"siddarthasamisetty.enterprise_de_project.{table}").count()
    logs.append((
        "enterprise_de_pipeline",
        layer,
        f"{layer}_record_count",
        start_time,
        datetime.now(),
        count,
        "SUCCESS",
        None
    ))

schema = spark.table("siddarthasamisetty.enterprise_de_project.pipeline_run_logs").schema
spark.createDataFrame(logs, schema) \
    .write.format("delta").mode("append") \
    .saveAsTable("siddarthasamisetty.enterprise_de_project.pipeline_run_logs")

# Validate logs
display(
    spark.table("siddarthasamisetty.enterprise_de_project.pipeline_run_logs")
    .orderBy("start_time", ascending=False)
)



pipeline_name,layer,step_name,start_time,end_time,record_count,status,error_message
enterprise_de_pipeline,gold,gold_record_count,2025-12-22T13:42:06.230Z,2025-12-22T13:42:06.825Z,30,SUCCESS,null
enterprise_de_pipeline,silver,silver_record_count,2025-12-22T13:42:06.230Z,2025-12-22T13:42:06.638Z,803,SUCCESS,null
enterprise_de_pipeline,bronze,bronze_record_count,2025-12-22T13:42:06.230Z,2025-12-22T13:42:06.449Z,1000,SUCCESS,null
enterprise_de_pipeline,silver,quarantine_record_count,2025-12-22T13:41:49.178Z,2025-12-22T13:41:49.178Z,203,SUCCESS,null
enterprise_de_pipeline,silver,rerun_safety_validation,2025-12-22T13:41:25.071Z,2025-12-22T13:41:29.127Z,803,SUCCESS,null
enterprise_de_pipeline,silver,quarantine_record_count,2025-12-22T13:39:56.589Z,2025-12-22T13:39:56.589Z,203,SUCCESS,null
enterprise_de_pipeline,silver,silver_record_count,2025-12-22T13:39:24.927Z,2025-12-22T13:39:25.430Z,803,SUCCESS,null
enterprise_de_pipeline,gold,gold_record_count,2025-12-22T13:39:24.927Z,2025-12-22T13:39:25.652Z,30,SUCCESS,null
enterprise_de_pipeline,bronze,bronze_record_count,2025-12-22T13:39:24.927Z,2025-12-22T13:39:25.220Z,1000,SUCCESS,null
enterprise_de_pipeline,silver,silver_sales_processing,2025-12-22T13:38:59.266Z,2025-12-22T13:38:59.897Z,803,SUCCESS,null


PHASE 8 — STEP 8.4
LOG REJECTED (QUARANTINE) RECORDS

In [0]:
from datetime import datetime

count = spark.table(
    "siddarthasamisetty.enterprise_de_project.silver_sales_quarantine"
).count()

spark.createDataFrame([(
    "enterprise_de_pipeline",
    "silver",
    "quarantine_record_count",
    datetime.now(),
    datetime.now(),
    count,
    "SUCCESS",
    None
)], spark.table("siddarthasamisetty.enterprise_de_project.pipeline_run_logs").schema) \
.write.format("delta").mode("append") \
.saveAsTable("siddarthasamisetty.enterprise_de_project.pipeline_run_logs")


# Validate logs
display(
    spark.table("siddarthasamisetty.enterprise_de_project.pipeline_run_logs")
    .orderBy("start_time", ascending=False)
)



pipeline_name,layer,step_name,start_time,end_time,record_count,status,error_message
enterprise_de_pipeline,silver,quarantine_record_count,2025-12-22T13:41:49.178Z,2025-12-22T13:41:49.178Z,203,SUCCESS,null
enterprise_de_pipeline,silver,rerun_safety_validation,2025-12-22T13:41:25.071Z,2025-12-22T13:41:29.127Z,803,SUCCESS,null
enterprise_de_pipeline,silver,quarantine_record_count,2025-12-22T13:39:56.589Z,2025-12-22T13:39:56.589Z,203,SUCCESS,null
enterprise_de_pipeline,silver,silver_record_count,2025-12-22T13:39:24.927Z,2025-12-22T13:39:25.430Z,803,SUCCESS,null
enterprise_de_pipeline,bronze,bronze_record_count,2025-12-22T13:39:24.927Z,2025-12-22T13:39:25.220Z,1000,SUCCESS,null
enterprise_de_pipeline,gold,gold_record_count,2025-12-22T13:39:24.927Z,2025-12-22T13:39:25.652Z,30,SUCCESS,null
enterprise_de_pipeline,silver,silver_sales_processing,2025-12-22T13:38:59.266Z,2025-12-22T13:38:59.897Z,803,SUCCESS,null


PHASE 8 — STEP 8.5
PIPELINE RERUN SAFETY & DUPLICATE PREVENTION

The pipeline is idempotent. Delta MERGE is used for Silver tables,
and Gold tables are rebuilt using overwrite logic. This ensures
safe reruns without data duplication.


In [0]:
# ============================================================
# PHASE 8 — STEP 8.5: PIPELINE RERUN SAFETY & DUPLICATE PREVENTION
# ============================================================

from datetime import datetime
from pyspark.sql.types import (
    StructType, StructField,
    StringType, TimestampType, LongType
)

pipeline_name = "enterprise_de_pipeline"
layer = "silver"
step_name = "rerun_safety_validation"

start_time = datetime.now()

try:
    # Re-run MERGE on same Silver table (safe rerun)
    spark.sql("""
        MERGE INTO siddarthasamisetty.enterprise_de_project.silver_sales_transactions AS target
        USING siddarthasamisetty.enterprise_de_project.silver_sales_transactions AS source
        ON target.transaction_id = source.transaction_id
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)
    
    # Record count remains unchanged → no duplicates
    record_count = spark.table(
        "siddarthasamisetty.enterprise_de_project.silver_sales_transactions"
    ).count()

    status = "SUCCESS"
    error_message = None

except Exception as e:
    record_count = None
    status = "FAILED"
    error_message = str(e)

end_time = datetime.now()

# Schema from logging table
log_schema = spark.table(
    "siddarthasamisetty.enterprise_de_project.pipeline_run_logs"
).schema

log_df = spark.createDataFrame([(
    pipeline_name,
    layer,
    step_name,
    start_time,
    end_time,
    record_count,
    status,
    error_message
)], schema=log_schema)

# Append rerun log
(
    log_df.write
    .format("delta")
    .mode("append")
    .saveAsTable("siddarthasamisetty.enterprise_de_project.pipeline_run_logs")
)

# Validate logs
display(
    spark.table("siddarthasamisetty.enterprise_de_project.pipeline_run_logs")
    .orderBy("start_time", ascending=False)
)


pipeline_name,layer,step_name,start_time,end_time,record_count,status,error_message
enterprise_de_pipeline,silver,rerun_safety_validation,2025-12-22T13:41:25.071Z,2025-12-22T13:41:29.127Z,803,SUCCESS,null
enterprise_de_pipeline,silver,quarantine_record_count,2025-12-22T13:39:56.589Z,2025-12-22T13:39:56.589Z,203,SUCCESS,null
enterprise_de_pipeline,silver,silver_record_count,2025-12-22T13:39:24.927Z,2025-12-22T13:39:25.430Z,803,SUCCESS,null
enterprise_de_pipeline,bronze,bronze_record_count,2025-12-22T13:39:24.927Z,2025-12-22T13:39:25.220Z,1000,SUCCESS,null
enterprise_de_pipeline,gold,gold_record_count,2025-12-22T13:39:24.927Z,2025-12-22T13:39:25.652Z,30,SUCCESS,null
enterprise_de_pipeline,silver,silver_sales_processing,2025-12-22T13:38:59.266Z,2025-12-22T13:38:59.897Z,803,SUCCESS,null
